# Probability in Machine Learning

Why has this course spent so much time on probabilities? Because **machine learning is a lot about probabilities** — most classifiers *are* conditional-probability calculators, and generative models work by maximizing probabilities. This conceptual notebook connects everything from Lesson 1 to machine learning. It is based on the fifteenth video of Week 1 (Introduction to Probability).


## 1. The big idea

In machine learning you very often want the **probability of something given some other factors** — a conditional probability:

| Task | Conditional probability computed |
|---|---|
| Spam detection | $P(\text{spam} \mid \text{words, recipients, attachments, ...})$ |
| Sentiment analysis | $P(\text{happy} \mid \text{words in the text})$ |
| Image recognition | $P(\text{cat} \mid \text{pixel}_1, \text{pixel}_2, \dots, \text{pixel}_n)$ |
| Medical diagnosis | $P(\text{healthy} \mid \text{symptoms, demographics, history})$ |

These are all **supervised machine learning**: answering a question about the data.

There is a second family — **generative machine learning** — where instead of *evaluating* a conditional probability, you *maximize* one:

- **Image generation**: choose pixels that maximize the probability that they form a human face (e.g. StyleGAN's photorealistic fake faces).
- **Text generation**: choose words that maximize the probability of sensical text about a topic.


## 2. The spam classifier, seen from high altitude

Everything from the Bayes notebooks was actually a machine-learning workflow:

1. **Prior** — measure $P(\text{spam})$ from the dataset. That's *training* on labeled data.
2. **Event/features** — the email contains "lottery". That's the *input*.
3. **Posterior** — $P(\text{spam} \mid \text{lottery})$ via the tree / Bayes formula. That's the *prediction*.

A classifier is exactly this: a machine that outputs $P(\text{label} \mid \text{evidence})$. The cat recognizer answers $P(\text{cat} \mid \text{pixels}) = 0.9$ for a cat photo and $0.1$ for a car photo; naive Bayes answers $P(\text{spam} \mid \text{words}) = 0.913$ for a lottery-winning email. Same shape, different features.


## 3. A toy sentiment classifier in ten lines

To make the point concrete, here is the naive Bayes machinery from notebook 14 re-aimed at **sentiment analysis** — nothing changes except the labels and features.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Word likelihoods measured from a (fictional) labeled dataset of sentences:
# P(word | happy), P(word | sad)
WORDS = {
    "wonderful": (0.30, 0.02),
    "love":      (0.25, 0.05),
    "awful":     (0.02, 0.35),
    "tears":     (0.04, 0.20),
    "sunshine":  (0.15, 0.03),
}
P_HAPPY = 0.5   # prior: half the sentences in the dataset are happy

def p_happy_given(words):
    happy, sad = P_HAPPY, 1 - P_HAPPY
    for w in words:
        p_w_happy, p_w_sad = WORDS[w]
        happy *= p_w_happy
        sad *= p_w_sad
    return happy / (happy + sad)

for sentence in [["love", "sunshine"], ["awful", "tears"], ["love", "awful"]]:
    p = p_happy_given(sentence)
    label = "HAPPY" if p > 0.5 else "SAD"
    print(f"{' + '.join(sentence):22s} P(happy | words) = {p:.3f} -> {label}")


## 4. Visualization: the classifier as a probability machine


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

def box(x, y, w, h, text, color):
    ax.add_patch(plt.Rectangle((x, y), w, h, facecolor=color,
                               edgecolor="black", linewidth=1.5))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=11)

# Supervised: evidence in, conditional probability out
box(0.0, 2.6, 2.4, 1.1, "evidence\n(pixels, words,\nsymptoms)", "#d8e6f2")
box(3.4, 2.6, 3.0, 1.1, "model\ncomputes\nP(label | evidence)", "#7fc97f")
box(7.4, 2.6, 2.4, 1.1, "prediction\nP(cat | pixels) = 0.9", "#f1ce63")
ax.annotate("", xy=(3.4, 3.15), xytext=(2.4, 3.15),
            arrowprops=dict(arrowstyle="->", lw=2))
ax.annotate("", xy=(7.4, 3.15), xytext=(6.4, 3.15),
            arrowprops=dict(arrowstyle="->", lw=2))
ax.text(-0.2, 3.9, "Supervised ML — evaluate a conditional probability",
        fontsize=12, fontweight="bold")

# Generative: maximize the probability
box(0.0, 0.2, 2.4, 1.1, "goal\n(a face, a story)", "#d8e6f2")
box(3.4, 0.2, 3.0, 1.1, "model\nmaximizes\nP(face | pixels)", "#b07aa1")
box(7.4, 0.2, 2.4, 1.1, "generated output\n(StyleGAN face)", "#f1ce63")
ax.annotate("", xy=(3.4, 0.75), xytext=(2.4, 0.75),
            arrowprops=dict(arrowstyle="->", lw=2))
ax.annotate("", xy=(7.4, 0.75), xytext=(6.4, 0.75),
            arrowprops=dict(arrowstyle="->", lw=2))
ax.text(-0.2, 1.5, "Generative ML — maximize a probability",
        fontsize=12, fontweight="bold")

ax.set_xlim(-0.4, 10.2)
ax.set_ylim(-0.2, 4.4)
ax.axis("off")
ax.set_title("Two ways machine learning uses probability")
plt.show()


### The whole story on one page

Everything above in a single sheet: supervised versus generative uses of probability, the toy sentiment classifier's three verdicts, each word's likelihood ratio, and how prior × likelihoods gives 0.222 for "love + awful".

<img src="img/15_probability_in_machine_learning.png" alt="Concept sheet for this notebook" width="900">

## 5. Exercises

1. Phrase a music-recommendation system as a conditional probability. What is the label, and what is the evidence?
2. In the toy sentiment classifier, "love" + "awful" gave a probability near 0.22 — clearly sad. Which single word could you add to the vocabulary and the sentence to flip it decisively happy, and why?
3. Is a weather app that says "70% chance of rain tomorrow" doing supervised or generative modeling in the video's sense?


## 6. Solutions

1. $P(\text{user likes this song} \mid \text{listening history, skips, likes, time of day})$ — the label is "likes the song", the evidence is the user's behavior features.
2. Any word with a large happy/sad likelihood ratio, e.g. "wonderful" (0.30 vs 0.02, a 15× ratio): multiplying it in raises the happy term far more than the sad one.
3. Supervised in this framing: it evaluates $P(\text{rain tomorrow} \mid \text{today's atmospheric measurements})$ — a conditional probability of a label given evidence, not the generation of new data.


## 7. Conclusion

- Supervised classifiers compute **conditional probabilities**: $P(\text{label} \mid \text{evidence})$ — spam given words, cat given pixels, diagnosis given symptoms.
- Generative models **maximize probabilities**: pick pixels/words that make $P(\text{face})$ or $P(\text{sensible text})$ as high as possible.
- The Bayes pipeline you built — prior, event, posterior — *is* a small machine-learning classifier; bigger models change the features and the machinery, not the underlying question.
- This closes Lesson 1. Lesson 2 introduces **probability distributions**, the language for describing whole families of probabilities at once.
